In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm import tqdm

from collections import Counter
from collections import defaultdict

from albumentations.pytorch import ToTensorV2
import albumentations as A

import cv2
import numpy as np
import timm

import random
import os
from glob import glob

d:\LabsMIET\MLlab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 9999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [3]:
class_to_idx = { "Апельсин": 0,
                 "Бананы": 1,
                 "Груши": 2, 
                 "Кабачки": 3, 
                 "Капуста": 4, 
                 "Картофель": 5, 
                 "Киви": 6, 
                 "Лимон": 7, 
                 "Лук": 8, 
                 "Мандарины": 9, 
                 "Морковь": 10, 
                 "Огурцы": 11, 
                 "Томаты": 12, 
                 "Яблоки зелёные": 13, 
                 "Яблоки красные": 14 }

In [4]:
class MyDataset(Dataset):
    def __init__(self, images_filepaths, name2label, transform=None):
        self.images_filepaths = images_filepaths
        self.transform = transform
        self.name2label = name2label

    def __len__(self):
        return len(self.images_filepaths)

    def __getitem__(self, idx):
        image_filepath = self.images_filepaths[idx]
        image = cv2.imdecode(np.fromfile(image_filepath, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.name2label[os.path.normpath(image_filepath).split(os.sep)[-3]]
        
        if self.transform is not None:
            image = self.transform(image=image)['image']
        return image, label


def train_test_split_from_directory(root_path, folder2class, train_size=0.8):
    train, test = [], []

    for class_name in os.listdir(root_path):
        class_path = os.path.join(root_path, class_name)
        if not os.path.isdir(class_path):
            continue

        for subclass_name in os.listdir(class_path):
            subclass_path = os.path.join(class_path, subclass_name)
            if not os.path.isdir(subclass_path):
                continue

            images = glob(os.path.join(subclass_path, '*.jpg')) + \
                     glob(os.path.join(subclass_path, '*.png')) + \
                     glob(os.path.join(subclass_path, '*.jpeg'))
            
            if len(images) == 0:
                continue
            
            # делим подклассы в пропорции 80/20
            random.shuffle(images)
            split_idx = int(train_size * len(images))

            if split_idx == 0 and len(images) > 0:
                split_idx = 1

            train.extend(images[:split_idx])
            test.extend(images[split_idx:])

    random.shuffle(train)
    random.shuffle(test)

    return train, test

Настройка датасета, writer для вывода, device - на чем обучается

In [5]:
dataset_path = 'train/train'
train, test = train_test_split_from_directory(dataset_path, class_to_idx)

writer = SummaryWriter("kirillLogs")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Функция ошибки

In [6]:
class FocalSmoothingLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, logits, targets):
        n_classes = logits.size(-1)
        
        # Label smoothing  
        with torch.no_grad():
            true_dist = torch.full_like(logits, self.label_smoothing / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        
        # Log softmax 
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Cross entropy с label smoothing
        ce_loss = -(true_dist * log_probs).sum(dim=-1)        
        
        # Focal-часть 
        pt = torch.exp(-ce_loss)                                 
        modulating_factor = (1 - pt) ** self.gamma
        
        #  Class weights 
        if self.alpha is not None:
            if self.alpha.dim() == 1:  
                alpha_t = self.alpha[targets]
            else:
                alpha_t = self.alpha
            loss = alpha_t * modulating_factor * ce_loss
        else:
            loss = modulating_factor * ce_loss
        
        # Редукция
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

Модель

In [7]:
class HierarchicalSwinV2(nn.Module):
    def __init__(self, num_classes=15, model_name="swinv2_cr_small_ns_224.sw_in1k"):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True  
        )

        feature_channels = self.backbone.feature_info.channels()

        self.heads = nn.ModuleList([
            nn.Sequential(  # stage 1 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[0], 96),
                nn.ReLU(),
            ),
            nn.Sequential(  # stage 2
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[1], 128),
                nn.ReLU(),
            ),
            nn.Sequential(  # stage 3 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[2], 192),
                nn.ReLU(),
                nn.Dropout(0.15),
            ),
            nn.Sequential(  # stage 4 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[3], 256),
                nn.ReLU(),
                nn.Dropout(0.2),
            ),
        ])
        
        self.classifier = nn.Sequential(
            nn.Linear(672, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x) 
    
        pooled = []
        for i in range(4):
            feat = features[i]
            out = self.heads[i](feat)        
            pooled.append(out)
    
        fused = torch.cat(pooled, dim=1)     
    
        logits = self.classifier(fused)
        return logits

Классы, классы весов, sampling

Аугментация

In [8]:
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),

    A.Affine( # Масштаб, композиция, поворот
        shift_limit=0.1,
        scale_limit=0.15,
        rotate_limit=30,
        border_mode=0,          
        value=0,
        p=0.6
    ),

    A.ColorJitter( # Гамма
        brightness=0.25,
        contrast=0.25, 
        saturation=0.25, 
        hue=0.0, 
        p=0.5
    ),

    A.RandomShadow(p=0.25), # Тени
    A.RandomFog(p=0.12, fog_coef_lower=0.1, fog_coef_upper=0.4), # Туман
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2), # Зернистость

    A.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

C:\Users\Kirill\AppData\Local\Temp\ipykernel_23304\3643452720.py:5: UserWarning: Argument(s) 'shift_limit, scale_limit, rotate_limit, value' are not valid for transform Affine
  A.Affine( # Масштаб, композиция, поворот
C:\Users\Kirill\AppData\Local\Temp\ipykernel_23304\3643452720.py:23: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(p=0.12, fog_coef_lower=0.1, fog_coef_upper=0.4), # Туман
C:\Users\Kirill\AppData\Local\Temp\ipykernel_23304\3643452720.py:24: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2), # Зернистость


Датасеты, лоадеры

In [9]:
train_dataset = MyDataset(
    images_filepaths=train, 
    name2label=class_to_idx, 
    transform=train_transforms
)
test_dataset = MyDataset(
    images_filepaths=test, 
    name2label=class_to_idx, 
    transform=val_transforms
)

In [10]:
def count_classes(image_paths, class_to_idx):
    counts = Counter()
    for path in image_paths:
        class_name = os.path.normpath(path).split(os.sep)[-3]
        class_idx = class_to_idx[class_name]
        counts[class_idx] += 1
    return counts

train_counts = count_classes(train, class_to_idx)
test_counts  = count_classes(test, class_to_idx)

total_samples = sum(train_counts.values())         
num_classes = len(train_counts)

class_weights = torch.tensor(
    [total_samples / (num_classes * train_counts[i]) for i in range(num_classes)],
    dtype=torch.float32
)

class_weights = class_weights / class_weights.mean()

# train_labels = torch.tensor([label for _, label in train_dataset])

# weigth_sampling = class_weights[train_labels]

# sampler_weigths = WeightedRandomSampler(
#     weights=weigth_sampling,
#     num_samples=len(weigth_sampling),
#     replacement=True
# )


In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    # sampler=sampler_weigths,
    shuffle=True,
    num_workers=0,  
    pin_memory=True,  
    persistent_workers=False  
)
test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,  
    pin_memory=True,  
    persistent_workers=False
)

Функция для обучения

In [ ]:
# import torch
# from tqdm import tqdm

# @torch.no_grad()
# def evaluate(model, dataloader, loss_fn, device, desc="Val"):
#     model.eval()

#     total_loss = 0.0
#     total_correct = 0
#     total_samples = 0

#     pbar = tqdm(dataloader, desc=desc, leave=False)
#     for images, labels in pbar:
#         images = images.to(device)
#         labels = labels.to(device)

#         logits = model(images)
#         loss = loss_fn(logits, labels)

#         batch_size = labels.size(0)
#         total_loss += loss.item() * batch_size

#         y_pred = logits.argmax(dim=1)
#         total_correct += (y_pred == labels).sum().item()
#         total_samples += batch_size

#         avg_loss = total_loss / max(total_samples, 1)
#         acc = total_correct / max(total_samples, 1)
#         pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{acc:.4f}")

#     avg_loss = total_loss / max(total_samples, 1)
#     accuracy = total_correct / max(total_samples, 1)
#     return accuracy, avg_loss


# def train(model, criterion, optimizer, scheduler, train_loader, val_loader, device, writer=None, n_epochs=5):
#     num_iter = 0

#     for epoch in range(1, n_epochs + 1):
#         model.train()

#         total_loss = 0.0
#         total_correct = 0
#         total_samples = 0

#         pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}", leave=True)

#         for images, labels in pbar:
#             images = images.to(device)
#             labels = labels.to(device)

#             logits = model(images)
#             loss = criterion(logits, labels)

#             optimizer.zero_grad(set_to_none=True)
#             loss.backward()
            
#             optimizer.step()
#             scheduler.step()

#             # Накопим метрики для прогресс-бара
#             batch_size = labels.size(0)
#             total_loss += loss.item() * batch_size
#             total_samples += batch_size

#             y_pred = logits.argmax(dim=1)
#             total_correct += (y_pred == labels).sum().item()

#             avg_loss = total_loss / max(total_samples, 1)
#             acc = total_correct / max(total_samples, 1)

#             # tqdm live-metrics
#             pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

#             # Логирование (по итерациям)
#             num_iter += 1
#             if writer is not None:
#                 writer.add_scalar("Loss/train", loss.item(), num_iter)
#                 writer.add_scalar("Accuracy/train", (y_pred == labels).float().mean().item(), num_iter)

#         # Валидация (тоже с tqdm)
#         val_acc, val_loss = evaluate(model, val_loader, criterion, device, desc=f"Val {epoch}/{n_epochs}")

#         if writer is not None:
#             writer.add_scalar("Loss/val", val_loss, num_iter)
#             writer.add_scalar("Accuracy/val", val_acc, num_iter)

#         print(f"Epoch {epoch}/{n_epochs}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

#     return model

Инициализация модели

In [ ]:
# import torch
# import torch.optim as optim

# n_epochs = 15

# model = HierarchicalSwinV2(num_classes=15).to(device)

# criterion =  FocalSmoothingLoss(
#     alpha=class_weights.to(device),
#     gamma=2.0,
#     label_smoothing=0.05
# )

# optimizer = optim.AdamW(model.parameters(), weight_decay=1e-2)

# scheduler = torch.optim.lr_scheduler.OneCycleLR(
#     optimizer,
#     max_lr=1e-3,                # пик lr
#     epochs=n_epochs,
#     steps_per_epoch=len(train_loader),
#     pct_start=0.3,              # 30% времени на рост
#     anneal_strategy='cos',
#     div_factor=30.0,
#     final_div_factor=1e4
# )

Обучение

In [ ]:

# model = train(
#     model, 
#     criterion, 
#     optimizer, 
#     scheduler,
#     train_loader, 
#     test_loader,  
#     device, 
#     writer, 
#     n_epochs=15
# )


Epoch 1/15: 100%|██████████| 976/976 [02:53<00:00,  5.62it/s, train_acc=0.5790, train_loss=0.8707]


Epoch 1/15: val_loss=0.5160  val_acc=0.7139


Epoch 2/15: 100%|██████████| 976/976 [02:53<00:00,  5.62it/s, train_acc=0.5991, train_loss=0.7757]


Epoch 2/15: val_loss=0.7094  val_acc=0.6154


Epoch 3/15: 100%|██████████| 976/976 [02:55<00:00,  5.55it/s, train_acc=0.5558, train_loss=0.8257]


...

Epoch 13/15: val_loss=0.2207  val_acc=0.8808


Epoch 14/15: 100%|██████████| 976/976 [03:05<00:00,  5.27it/s, train_acc=0.8637, train_loss=0.2353]


Epoch 14/15: val_loss=0.2082  val_acc=0.8838


Epoch 15/15: 100%|██████████| 976/976 [03:02<00:00,  5.34it/s, train_acc=0.8791, train_loss=0.2151]
                                                                                     

Epoch 15/15: val_loss=0.2055  val_acc=0.8869


Тест

In [12]:
@torch.no_grad()
def evaluate(model, dataloader, loss_fn, device, desc="Val"):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(dataloader, desc=desc, leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        y_pred = logits.argmax(dim=1)
        total_correct += (y_pred == labels).sum().item()
        total_samples += batch_size

        avg_loss = total_loss / max(total_samples, 1)
        acc = total_correct / max(total_samples, 1)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{acc:.4f}")

    avg_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)
    return accuracy, avg_loss

def training_with_stage(model, criterion, train_loader, val_loader, device, writer, n_epoch=25):
    num_iter = 0
    print('Stage 1', n_epoch)

    head_params = [p for n, p in model.named_parameters() 
                   if "heads" in n or "classifier" in n]

    optimizer_stage1 = optim.AdamW(head_params, lr=1e-3, weight_decay=0.05)

    scheduler_stage1 = optim.lr_scheduler.OneCycleLR(
        optimizer_stage1,
        max_lr=1e-3,
        epochs=5,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos',
        div_factor=25.0,
        final_div_factor=1e4
    )


    for epoch in range(1, 6):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epoch}", leave=True)

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            optimizer_stage1.zero_grad(set_to_none=True)
            loss.backward()
            
            optimizer_stage1.step()
            scheduler_stage1.step()
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == labels).sum().item()

            avg_loss = total_loss / max(total_samples, 1)
            acc = total_correct / max(total_samples, 1)

            # tqdm live-metrics
            pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

            # Логирование (по итерациям)
            num_iter += 1
            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), num_iter)
                writer.add_scalar("Accuracy/train", (y_pred == labels).float().mean().item(), num_iter)

        # Валидация (тоже с tqdm)
        val_acc, val_loss = evaluate(model, val_loader, criterion, device, desc=f"Val {epoch}/{n_epoch}")

        if writer is not None:
            writer.add_scalar("Loss/val", val_loss, num_iter)
            writer.add_scalar("Accuracy/val", val_acc, num_iter)

        print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")


    print('Stage 2')
    for param in model.backbone.parameters():
        param.requires_grad = True

    optimizer_stage2 = optim.AdamW([
        {'params': [p for n, p in model.named_parameters() if "backbone" in n],
         'lr': 5e-6, 'weight_decay': 0.05},
        {'params': [p for n, p in model.named_parameters() if "heads" in n or "classifier" in n],
         'lr': 5e-5, 'weight_decay': 0.05},
    ])

    scheduler_stage2 = torch.optim.lr_scheduler.OneCycleLR(
        optimizer_stage2,
        max_lr=[5e-6, 5e-5],               
        epochs=n_epoch - 5,         
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )


    for epoch in range(6, n_epoch + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epoch}", leave=True)

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            optimizer_stage2.zero_grad(set_to_none=True)
            loss.backward()
            
            optimizer_stage2.step()
            scheduler_stage2.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == labels).sum().item()

            avg_loss = total_loss / max(total_samples, 1)
            acc = total_correct / max(total_samples, 1)

            # tqdm live-metrics
            pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

            # Логирование (по итерациям)
            num_iter += 1
            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), num_iter)
                writer.add_scalar("Accuracy/train", (y_pred == labels).float().mean().item(), num_iter)

        # Валидация (тоже с tqdm)
        val_acc, val_loss = evaluate(model, val_loader, criterion, device, desc=f"Val {epoch}/{n_epoch}")

        if writer is not None:
            writer.add_scalar("Loss/val", val_loss, num_iter)
            writer.add_scalar("Accuracy/val", val_acc, num_iter)

        print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    return model

In [13]:
n_epochs = 30

model = HierarchicalSwinV2(num_classes=15).to(device)

params_stage1 = []
for name, param in model.named_parameters():
    if "heads" in name or "classifier" in name:
        params_stage1.append({
            'params': param,
            'lr': 1e-3,
            'weight_decay': 0.05
        })
    else:
        param.requires_grad = False

criterion = FocalSmoothingLoss(
    alpha=class_weights.to(device),
    gamma=2.0,
    label_smoothing=0.1
)

In [14]:
model = training_with_stage(
    model,
    criterion,
    train_loader,
    test_loader,
    device,
    writer,
    n_epochs
)

Stage 1 30


Epoch 1/30: 100%|██████████| 976/976 [01:50<00:00,  8.87it/s, train_acc=0.4689, train_loss=1.2367]


Epoch 1/30: val_loss=0.5480  val_acc=0.7879


Epoch 2/30: 100%|██████████| 976/976 [01:54<00:00,  8.49it/s, train_acc=0.6901, train_loss=0.7057]


Epoch 2/30: val_loss=0.5021  val_acc=0.7823


...

Epoch 28/30: val_loss=0.2035  val_acc=0.9569


Epoch 29/30: 100%|██████████| 976/976 [04:40<00:00,  3.48it/s, train_acc=0.9927, train_loss=0.1325]


Epoch 29/30: val_loss=0.2043  val_acc=0.9564


Epoch 30/30: 100%|██████████| 976/976 [04:39<00:00,  3.49it/s, train_acc=0.9912, train_loss=0.1338]
                                                                                     

Epoch 30/30: val_loss=0.2043  val_acc=0.9564


In [15]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

@torch.no_grad()
def sklearn_report(model, dataloader, device, idx2class=None, digits=4):
    model.eval()

    y_true, y_pred = [], []

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)

        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        y_pred.append(preds)
        y_true.append(labels.numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    if idx2class is None:
        target_names = None
        labels = None
    else:
        labels = sorted(idx2class.keys())
        target_names = [idx2class[i] for i in labels]

    rep = classification_report(
        y_true, y_pred,
        labels=labels,
        target_names=target_names,
        digits=digits,
        zero_division=0
    )
    print(rep)

    if idx2class is not None:
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print("\nConfusion Matrix:")
        print(cm)

In [16]:
idx2class = {v: k for k, v in class_to_idx.items()}

sklearn_report(model, test_loader, device, idx2class=idx2class, digits=4)

                precision    recall  f1-score   support

      Апельсин     0.9611    0.9774    0.9692       177
        Бананы     0.9756    0.9877    0.9816       162
         Груши     0.9595    0.8765    0.9161        81
       Кабачки     0.9194    0.9661    0.9421        59
       Капуста     1.0000    0.9702    0.9849       168
     Картофель     0.9613    0.9430    0.9521       158
          Киви     0.8462    0.8684    0.8571        38
         Лимон     0.9845    0.9549    0.9695       133
           Лук     0.9037    0.9242    0.9139       132
     Мандарины     0.9494    0.9615    0.9554       156
       Морковь     0.9675    0.9597    0.9636       124
        Огурцы     0.9587    0.9915    0.9748       117
        Томаты     0.9481    0.9733    0.9605       150
Яблоки зелёные     0.9540    0.9765    0.9651       170
Яблоки красные     0.9500    0.9110    0.9301       146

      accuracy                         0.9564      1971
     macro avg     0.9493    0.9495    0.9491 

In [19]:
test_images_dir = "test_images/test_images"
submission_path = "sample_submission.csv"
output_path = "submissionKirill.csv"

In [20]:
import pandas as pd
submission = pd.read_csv(submission_path)

model.eval()
pred_labels = []

with torch.no_grad():
    for image_id in tqdm(submission["image_id"], desc="Predicting"):
        image_path = os.path.join(test_images_dir, image_id)

        image = cv2.imdecode(
            np.fromfile(image_path, dtype=np.uint8),
            cv2.IMREAD_COLOR
        )
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Когда мы хотим сделать предсказание, нам нужно знать, а какие были преобразования при обучении/тестировании
        if val_transforms is not None:
            image = val_transforms(image=image)["image"]

        image = image.unsqueeze(0).to(device)

        logits = model(image)
        pred_idx = logits.argmax(dim=1).item()

        pred_labels.append(pred_idx)


Predicting: 100%|██████████| 2503/2503 [01:17<00:00, 32.29it/s]


In [21]:
submission["label"] = pred_labels
submission.to_csv(output_path, index=False)

submission.head()


,image_id,label
0,fd343552326b42c5a62c192f32549dc7.jpg,4
1,445ca69812cf44f581cc8a89223af277.jpg,7
2,570626ce4d8f41edb8088f49d40a2195.jpg,7
3,02d4acba92f343d798adcb4958fe684b.jpg,11
4,2d4b8e8f38534a39b0d02c440e917b83.jpg,12
